# EXPERIMENT 5
# Naive Bayes Classification & Model Evaluation

## Objective

- Implement Laplace smoothing to resolve the zero-frequency problem.
- Build a Naive Bayes classifier from scratch using Python and NumPy.
- Train and evaluate Scikit-learn's MultinomialNB and GaussianNB models.
- Compute and interpret evaluation metrics using a Confusion Matrix.

---

## Dataset

Consider the following historical email dataset classified into Spam
and Not Spam.

| Email ID | Features Present in Email | Class Label |
|----------|----------------------------|-------------|
| E1 | offer, winner | Spam |
| E2 | offer, bonus | Spam |
| E3 | winner, bonus | Spam |
| E4 | meeting, bonus | Not Spam |
| E5 | meeting, offer | Not Spam |

### Test Instance

A new email arrives with:

$$
X = \{offer,\ bonus\}
$$

We need to determine whether the email is:

- Spam
- Not Spam

using Naive Bayes with Laplace smoothing.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

# Historical email dataset
data = {
    "Email ID": ["E1", "E2", "E3", "E4", "E5"],
    "Features": [
        ["offer", "winner"],
        ["offer", "bonus"],
        ["winner", "bonus"],
        ["meeting", "bonus"],
        ["meeting", "offer"]
    ],
    "Class": ["Spam", "Spam", "Spam", "Not Spam", "Not Spam"]
}

df = pd.DataFrame(data)

print("Historical Email Dataset:")
display(df)


Historical Email Dataset:


,Email ID,Features,Class
0,E1,"[offer, winner]",Spam
1,E2,"[offer, bonus]",Spam
2,E3,"[winner, bonus]",Spam
3,E4,"[meeting, bonus]",Not Spam
4,E5,"[meeting, offer]",Not Spam


## Naive Bayes Classification from Scratch

Naive Bayes uses Bayes' theorem:

$$
P(C|X) \propto P(C)P(X|C)
$$

For multiple features:

$$
P(C|X) \propto P(C)
\prod_{i=1}^{n}P(x_i|C)
$$

For the test email:

$$
X = \{offer,\ bonus\}
$$

### Step 1: Calculate Prior Probabilities

There are 5 emails:

- Spam = 3
- Not Spam = 2

Therefore,

$$
P(Spam)=\frac{3}{5}=0.6
$$

$$
P(NotSpam)=\frac{2}{5}=0.4
$$

### Step 2: Vocabulary

The complete vocabulary is:

$$
V=\{offer,winner,bonus,meeting\}
$$

Therefore:

$$
|V|=4
$$

### Step 3: Laplace Smoothing

The Laplace-smoothed probability is:

$$
P(word|class)=
\frac{Count(word,class)+1}
{TotalWords(class)+|V|}
$$

The `+1` prevents a probability from becoming zero.

In [3]:
# -----------------------------------------
# Naive Bayes from Scratch
# -----------------------------------------

# Classes
classes = ["Spam", "Not Spam"]

# Vocabulary
vocabulary = sorted(
    set(word for features in df["Features"] for word in features)
)

V = len(vocabulary)

print("Vocabulary:", vocabulary)
print("Vocabulary size:", V)

# Calculate class priors
class_counts = df["Class"].value_counts()

priors = {
    cls: class_counts[cls] / len(df)
    for cls in classes
}

print("\nClass Priors:")
for cls, prob in priors.items():
    print(f"P({cls}) = {prob:.4f}")

# Count words for each class
word_counts = {}

for cls in classes:

    word_counts[cls] = {}

    class_emails = df[df["Class"] == cls]

    for word in vocabulary:

        count = sum(
            word in features
            for features in class_emails["Features"]
        )

        word_counts[cls][word] = count

print("\nWord Counts:")
print(pd.DataFrame(word_counts))

Vocabulary: ['bonus', 'meeting', 'offer', 'winner']
Vocabulary size: 4

Class Priors:
P(Spam) = 0.6000
P(Not Spam) = 0.4000

Word Counts:
         Spam  Not Spam
bonus       2         1
meeting     0         2
offer       2         1
winner      2         0


### Step 4: Apply Laplace Smoothing

For the Spam class:

- Total word occurrences = 6
- Vocabulary size = 4

Therefore:

$$
Denominator = 6+4=10
$$

For `offer`:

$$
P(offer|Spam)
=
\frac{2+1}{10}
=
\frac{3}{10}
=
0.3
$$

For `bonus`:

$$
P(bonus|Spam)
=
\frac{2+1}{10}
=
0.3
$$

For the Not Spam class:

- Total word occurrences = 4
- Vocabulary size = 4

Therefore:

$$
Denominator = 4+4=8
$$

For `offer`:

$$
P(offer|NotSpam)
=
\frac{1+1}{8}
=
0.25
$$

For `bonus`:

$$
P(bonus|NotSpam)
=
\frac{1+1}{8}
=
0.25
$$

In [4]:
# Calculate total word occurrences for each class
total_words = {}

for cls in classes:
    total_words[cls] = sum(word_counts[cls].values())

print("Total word occurrences:")
for cls in classes:
    print(f"{cls}: {total_words[cls]}")

# Laplace-smoothed conditional probabilities
conditional_probs = {}

for cls in classes:

    conditional_probs[cls] = {}

    denominator = total_words[cls] + V

    for word in vocabulary:

        numerator = word_counts[cls][word] + 1

        conditional_probs[cls][word] = (
            numerator / denominator
        )

conditional_df = pd.DataFrame(conditional_probs)

print("\nLaplace-Smoothed Conditional Probabilities:")
display(conditional_df)

Total word occurrences:
Spam: 6
Not Spam: 4

Laplace-Smoothed Conditional Probabilities:


,Spam,Not Spam
bonus,0.3,0.250
meeting,0.1,0.375
offer,0.3,0.250
winner,0.3,0.125


Final Prediction

In [5]:
# Test email
test_email = ["offer", "bonus"]

# Calculate Naive Bayes scores
scores = {}

for cls in classes:

    probability = priors[cls]

    for word in test_email:
        probability *= conditional_probs[cls][word]

    scores[cls] = probability

print("Naive Bayes Scores:")

for cls, score in scores.items():
    print(f"{cls}: {score:.4f}")

# Normalize scores to obtain posterior probabilities
total_score = sum(scores.values())

posterior = {
    cls: scores[cls] / total_score
    for cls in classes
}

print("\nPosterior Probabilities:")

for cls, probability in posterior.items():
    print(f"P({cls}|X) = {probability:.4f}")

# Final prediction
prediction = max(
    posterior,
    key=posterior.get
)

print("\nFinal Prediction:", prediction)

Naive Bayes Scores:
Spam: 0.0540
Not Spam: 0.0250

Posterior Probabilities:
P(Spam|X) = 0.6835
P(Not Spam|X) = 0.3165

Final Prediction: Spam


### Final Calculation

For Spam:

$$
P(Spam|X) \propto
P(Spam)P(offer|Spam)P(bonus|Spam)
$$

$$
=0.6\times0.3\times0.3
$$

$$
=0.054
$$

For Not Spam:

$$
P(NotSpam|X) \propto
P(NotSpam)P(offer|NotSpam)P(bonus|NotSpam)
$$

$$
=0.4\times0.25\times0.25
$$

$$
=0.025
$$

Since:

$$
0.054 > 0.025
$$

the final prediction is:

# **Spam**

## Multinomial Naive Bayes using Scikit-Learn

The emails are converted into numerical feature vectors.

The vocabulary is:

$$
[bonus,\ meeting,\ offer,\ winner]
$$

Each email is represented using the presence or absence of these words.

We then train Scikit-learn's `MultinomialNB` classifier.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# Convert feature lists into text
df["Text"] = df["Features"].apply(lambda x: " ".join(x))

# Convert text into numerical features
vectorizer = CountVectorizer(
    vocabulary=vocabulary
)

X = vectorizer.fit_transform(df["Text"]).toarray()
y = df["Class"]

print("Vocabulary mapping:")
print(vectorizer.vocabulary_)

print("\nFeature Matrix:")
print(X)

print("\nTarget Classes:")
print(y.values)

Vocabulary mapping:
{'bonus': 0, 'meeting': 1, 'offer': 2, 'winner': 3}

Feature Matrix:
[[0 0 1 1]
 [1 0 1 0]
 [1 0 0 1]
 [1 1 0 0]
 [0 1 1 0]]

Target Classes:
['Spam' 'Spam' 'Spam' 'Not Spam' 'Not Spam']


In [8]:
# Train Multinomial Naive Bayes
mnb = MultinomialNB(alpha=1.0)

mnb.fit(X, y)

# Convert test email
X_test = vectorizer.transform(
    ["offer bonus"]
).toarray()

# Predict
mnb_prediction = mnb.predict(X_test)

print("Test Email: offer, bonus")
print("MultinomialNB Prediction:", mnb_prediction[0])

# Posterior probabilities
mnb_probabilities = mnb.predict_proba(X_test)

print("\nClass Probabilities:")

for cls, prob in zip(mnb.classes_, mnb_probabilities[0]):
    print(f"{cls}: {prob:.4f}")

Test Email: offer, bonus
MultinomialNB Prediction: Spam

Class Probabilities:
Not Spam: 0.3165
Spam: 0.6835


# Train Multinomial Naive Bayes
mnb = MultinomialNB(alpha=1.0)

mnb.fit(X, y)

# Convert test email
X_test = vectorizer.transform(
    ["offer bonus"]
).toarray()

# Predict
mnb_prediction = mnb.predict(X_test)

print("Test Email: offer, bonus")
print("MultinomialNB Prediction:", mnb_prediction[0])

# Posterior probabilities
mnb_probabilities = mnb.predict_proba(X_test)

print("\nClass Probabilities:")

for cls, prob in zip(mnb.classes_, mnb_probabilities[0]):
    print(f"{cls}: {prob:.4f}")

In [9]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load Iris dataset
iris = load_iris()

X_iris = iris.data
y_iris = iris.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_iris,
    y_iris,
    test_size=0.25,
    random_state=42,
    stratify=y_iris
)

# Create and train GaussianNB
gnb = GaussianNB()

gnb.fit(X_train, y_train)

# Prediction
y_pred = gnb.predict(X_test)

print("GaussianNB Accuracy:",
      accuracy_score(y_test, y_pred))

GaussianNB Accuracy: 0.9210526315789473


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load Iris dataset
iris = load_iris()

X_iris = iris.data
y_iris = iris.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_iris,
    y_iris,
    test_size=0.25,
    random_state=42,
    stratify=y_iris
)

# Create and train GaussianNB
gnb = GaussianNB()

gnb.fit(X_train, y_train)

# Prediction
y_pred = gnb.predict(X_test)

print("GaussianNB Accuracy:",
      accuracy_score(y_test, y_pred))

In [10]:
# Confusion Matrix for GaussianNB
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nEvaluation Metrics:")
print("-------------------")

print(
    f"Accuracy : {accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, y_pred, average='weighted'):.4f}"
)

print(
    f"F1 Score : {f1_score(y_test, y_pred, average='weighted'):.4f}"
)

Confusion Matrix:
[[12  0  0]
 [ 0 12  1]
 [ 0  2 11]]

Evaluation Metrics:
-------------------
Accuracy : 0.9211
Precision: 0.9226
Recall   : 0.9211
F1 Score : 0.9209


# Observations

- Naive Bayes uses Bayes' theorem to calculate the probability of
  each possible class.
- Laplace smoothing prevents zero probabilities when a word has not
  appeared in a particular class.
- For the given test email `{offer, bonus}`, the probability score
  for Spam is greater than the probability score for Not Spam.
- Therefore, the test email is classified as **Spam**.
- The manually implemented Naive Bayes classifier and Scikit-learn's
  MultinomialNB can be used to perform the same type of classification.
- MultinomialNB is appropriate for discrete word/count features in
  text classification.
- GaussianNB is suitable for numerical continuous features.
- The confusion matrix helps identify correct and incorrect
  classifications.
- Accuracy, Precision, Recall and F1 Score provide different views
  of model performance.

# Conclusion

Naive Bayes successfully classified the given test email
`{offer, bonus}` as **Spam** using Laplace-smoothed probabilities.

Laplace smoothing solved the zero-frequency problem and allowed
probabilities to be calculated even when a feature was absent from
one of the classes.